In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm


import project_images_loader as pil
import utils.util as util
import utils.util_freesurfer_segmentation as ufs
import utils.functions as fc
import utils.nifti_functions as nfc

import constants as cts


SERVER: False


In [2]:
def compute_min_max_mean_std(img):
    min_img = np.min(img)
    max_img = np.max(img)
    mean_img = np.mean(img)
    std_img = np.std(img)
    return {"min":min_img, "max":max_img, "mean":mean_img, "std":std_img}


def compute_df_means(df_dicts):
    mean_dict = {}
    for name, df in df_dicts.items():
        mean_dict[name] = df.mean()
    df_means = pd.DataFrame(mean_dict)
    return df_means.transpose()


def load_image(hunt, id, registered_hunt3=True, param_file="0038"):
    # in the case that we want to load the original image to compare the baseline
    if not registered_hunt3 or hunt == 4:
        img, _ = nfc.load_nifti(pil.get_registered_to_template_image_path_name(hunt, id, pil.REGISTRATION_TO_MNI, absolute=True))
        seg, _ = nfc.load_nifti(pil.get_segmentation_path_name(hunt, id, nb_tissues=96, absolute=True))
    else:
        img, _ = nfc.load_nifti(cts.get_image_path_name(id, param_file, cts.INAME_REGISTERED))
        seg, _ = nfc.load_nifti(cts.get_image_path_name(id, param_file, cts.INAME_SEG_REGISTERED, nb_tissues=96))
    img = util.skull_stripping_and_normalize(img, seg, percentile=(0,100))
    return img


def load_segmentation(hunt, id, registered_hunt3=False, param_file="0038"):
    if not registered_hunt3 or hunt == 4:
        seg, _ = nfc.load_nifti(pil.get_segmentation_path_name(hunt, id, nb_tissues=96, absolute=True))
    else:
        seg, _ = nfc.load_nifti(cts.get_image_path_name(id, param_file, cts.INAME_SEG_REGISTERED, nb_tissues=96))
    return seg


In [3]:
def visualize_probability_map():
    %matplotlib widget
    image_size=(193,229,193)
    img_sum = np.zeros((image_size), dtype=np.float32)
    matching_ids = pil.get_matching_ids()
    for id in tqdm(matching_ids[:100]):
        # Tu código aquí
        for hunt in pil.HUNTS:
            img = load_image(hunt, id)
            img_sum += img[:image_size[0], :image_size[1], :image_size[2]]
    fc.imgshow_3D(f"Registered", img_sum, current_plane=98)

# visualize_probability_map()

In [4]:
def visualize_pairwise(ids, registered_hunt3=True):
    %matplotlib widget
    # for id in [pil.get_matching_ids()[0]]:
    # for id in ["01388", "02408", "06611", "00381"]:
    img_list = []
    for id in ids:
        image_size = (193,229,193)
        img_hunt3 = load_image(3, id, registered_hunt3=registered_hunt3)
        img_hunt4 = load_image(4, id, registered_hunt3=registered_hunt3)
        img_diff = np.abs(img_hunt3 - img_hunt4)
        img_list.append(img_diff)
        # fc.imgshow_3D(f"Registered", img_diff, current_plane=90, cmap="hot")
    fc.imgshow_3D_list(f"Registered", img_list, subimg_titles=ids, current_plane=90, cmap="hot", figsize=(10,5))
# visualize_pairwise(["00039", "08202"], registered_hunt3=True)


In [5]:
def compute_pairwise_similarity_metrics(ids, registered_hunt3=True):
    metrics_list = []
    for id in tqdm(ids):
        img_HUNT3 = load_image(3, id, registered_hunt3=registered_hunt3)
        img_HUNT4 = load_image(4, id, registered_hunt3=registered_hunt3)

        metrics = util.get_metrics_dict(img_HUNT3, img_HUNT4)
        metrics_list.append(metrics)

    df = pd.DataFrame(metrics_list)
    return df

ids = pil.get_matching_ids()[:3]
df_img = compute_pairwise_similarity_metrics(ids, registered_hunt3=True)
# print(df_img.head(10))



100%|██████████| 3/3 [00:06<00:00,  2.10s/it]


In [8]:
def compute_pairwise_dice_segmentation(ids, registered_hunt3=True):
    metrics_list = []
    for id in tqdm(ids):
        seg_HUNT3 = load_segmentation(3, id, registered_hunt3=registered_hunt3)
        seg_HUNT4 = load_segmentation(4, id, registered_hunt3=registered_hunt3)
        seg_HUNT3 = ufs.merge_seg96_to_seg3(seg_HUNT3)
        seg_HUNT4 = ufs.merge_seg96_to_seg3(seg_HUNT4)
        metrics = util.compute_multiple_dice_coefficients(seg_HUNT3, seg_HUNT4)
        metrics_list.append(metrics)

    df = pd.DataFrame(metrics_list)
    return df

ids = pil.get_matching_ids()[:3]
df_seg = compute_pairwise_dice_segmentation(ids, registered_hunt3=True)

print(df_seg.head(10))



100%|██████████| 3/3 [00:05<00:00,  1.74s/it]

          0         1         2         3
0  0.994986  0.945507  0.889825  0.938054
1  0.993400  0.964556  0.878241  0.932752
2  0.994065  0.965446  0.881873  0.933073
